In [2]:
# =========================
# 0. Install dependencies
# =========================

!pip install -q transformers sentencepiece protobuf tiktoken scikit-learn pandas tqdm torch


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

if not os.path.exists("/content/AIR_Group_Task"):
    !git clone https://github.com/ayadssk/AIR_Group_Task.git
else:
    print("Repository already exists, skipping clone.")

Cloning into 'AIR_Group_Task'...
remote: Enumerating objects: 294, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 294 (delta 10), reused 0 (delta 0), pack-reused 254 (from 1)
Receiving objects: 100% (294/294), 43.73 MiB | 9.19 MiB/s, done.
Resolving deltas: 100% (157/157), done.
Updating files: 100% (73/73), done.


In [5]:
# =========================
# 1. Imports
# =========================

import re
import json
import sys
import shutil
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm
import time
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

In [6]:
# =========================
# 2. Paths and config
# =========================

BASE = "/content/drive/MyDrive/AIR_CheckThat"
os.chdir(BASE)

BASE_MODEL   = "roberta-base"
MAX_LENGTH   = 128
BATCH_SIZE   = 4
EPOCHS       = 3
LR           = 1e-5
RANDOM_STATE = 42

# =========================
# Metadata
# =========================

MODEL_NAME = "roberta"
MODEL_TYPE = "baseline"          # baseline / experiment
DISTILLATION_STRATEGY = "none"   # none / response_mse / contrastive / grouped_contrastive / feature_based
PREPROCESSING_FAMILY = "custom_evidence_preprocessing"
REPO_BASE = "/content/AIR_Group_Task"
SCORER_PATH = f"{REPO_BASE}/task2/scorer.py"
SCORER_IO_DIR = f"{BASE}/output/RM_prediction"
SCORER_INPUT = f"{SCORER_IO_DIR}/clef_predictions.json"
SCORER_RESULT = f"{SCORER_IO_DIR}/result.csv"
SCORER_IR = f"{SCORER_IO_DIR}/per_sample_ir.csv"
PREPROCESSOR_PATH = f"{REPO_BASE}/experiments/teacher_llama_evidence/reasoning_trace_build_with_evidence.py"
os.makedirs(SCORER_IO_DIR, exist_ok=True)

MODEL_ID = BASE_MODEL.replace("/", "_").replace("-", "_")
DISTILL_TAG = DISTILLATION_STRATEGY.replace(" ", "_").replace("-", "_")

# Main organized output directories
RESULTS_ROOT = f"{BASE}/output/results"
PRED_ROOT    = f"{BASE}/output/RM_prediction"
CKPT_ROOT    = f"{BASE}/output/checkpoints"

os.makedirs(RESULTS_ROOT, exist_ok=True)
os.makedirs(PRED_ROOT, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Working dir:", os.getcwd())
print("Device:", device)

# ── Language registry ─────────────────────────────────────────────────────
LANGUAGES = [
    {
        "lang"       : "english",
        "train_path" : f"{BASE}/data/english/english_train.json",
        "val_path"   : f"{BASE}/data/english/clef2026_gpt4_o_mini_val.json",
        "test_path"  : f"{BASE}/data/english/clef_2026_final_english_test.json",
        "train_jsonl": f"{BASE}/output/training_data_for_RM/english_train_with_evidence.jsonl",
    },
    {
        "lang"       : "spanish",
        "train_path" : f"{BASE}/data/spanish/spanish_train.json",
        "val_path"   : f"{BASE}/data/spanish/spanish_val.json",
        "test_path"  : f"{BASE}/data/spanish/clef_spanish_test_final.json",
        "train_jsonl": f"{BASE}/output/training_data_for_RM/spanish_train_with_evidence.jsonl",
    },
    {
        "lang"       : "arabic",
        "train_path" : f"{BASE}/data/arabic/clef2026_gpt4_o_mini_train_arabic.json",
        "val_path"   : f"{BASE}/data/arabic/clef2026_gpt4_o_mini_val_arabic.json",
        "test_path"  : f"{BASE}/data/arabic/clef_2026_final_arabic_test.json",
        "train_jsonl": f"{BASE}/output/training_data_for_RM/arabic_train_with_evidence.jsonl",
    },
]

# ── Evidence preprocessing condition ──────────────────────────────────────
# This notebook uses custom evidence preprocessing.
# Input format: Claim + Evidence + Verdict + Justification
EVIDENCE_CONDITIONS = [True]

# ── Aggregation strategies ────────────────────────────────────────────────
from collections import Counter

def agg_top1(verdict_list, score_list):
    return verdict_list[int(np.argmax(score_list))]

def agg_majority_top3(verdict_list, score_list):
    top_idx = np.argsort(score_list)[::-1][:min(3, len(verdict_list))]
    return Counter([verdict_list[i] for i in top_idx]).most_common(1)[0][0]

def agg_majority_top5(verdict_list, score_list):
    top_idx = np.argsort(score_list)[::-1][:min(5, len(verdict_list))]
    return Counter([verdict_list[i] for i in top_idx]).most_common(1)[0][0]

def agg_score_weighted(verdict_list, score_list):
    weights = torch.sigmoid(torch.tensor(score_list)).numpy()
    tally = {}
    for v, w in zip(verdict_list, weights):
        tally[v] = tally.get(v, 0.0) + float(w)
    return max(tally, key=tally.get)

AGGREGATIONS = [
    ("top1",           agg_top1),
    ("majority_top3",  agg_majority_top3),
    ("majority_top5",  agg_majority_top5),
    ("score_weighted", agg_score_weighted),
]

# ── Pre-create output dirs ────────────────────────────────────────────────
os.makedirs(f"{BASE}/output/training_data_for_RM", exist_ok=True)
os.makedirs("output/RM_prediction", exist_ok=True)

for lc in LANGUAGES:
    lang = lc["lang"]
    os.makedirs(f"{RESULTS_ROOT}/{lang}", exist_ok=True)
    os.makedirs(f"{PRED_ROOT}/{lang}", exist_ok=True)
    os.makedirs(f"{CKPT_ROOT}/{lang}", exist_ok=True)


# checks

print(f"\n{len(LANGUAGES)} language(s) × {len(EVIDENCE_CONDITIONS)} evidence condition(s) × {len(AGGREGATIONS)} aggregation(s)")
print(f"Total training runs : {len(LANGUAGES) * len(EVIDENCE_CONDITIONS)}")
print(f"Total scored runs   : {len(LANGUAGES) * len(EVIDENCE_CONDITIONS) * len(AGGREGATIONS)}")
for lc in LANGUAGES:
    print(f"  [{lc['lang']:8s}]  train={os.path.exists(lc['train_path'])}  val={os.path.exists(lc['val_path'])}")


Working dir: /content/drive/.shortcut-targets-by-id/1yyjNePYVQPgD6oKJzAlLiZd1WSgsNjrB/AIR_CheckThat
Device: cpu

3 language(s) × 1 evidence condition(s) × 4 aggregation(s)
Total training runs : 3
Total scored runs   : 12
  [english ]  train=True  val=True
  [spanish ]  train=True  val=True
  [arabic  ]  train=True  val=True


In [10]:
# =========================
# 4. Helper functions
# =========================

def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting|Supports|Refutes))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")


def get_evidence(sample):
    possible_keys = [
        "evidences",
        "evidence",
        "Evidence",
        "relevant_evidence",
        "Relevant_evidence",
        "context",
        "Context",
        "gold_evidence",
        "Gold_evidence",
    ]

    for key in possible_keys:
        if key in sample and sample[key]:
            value = sample[key]

            if isinstance(value, list):
                return " ".join(map(str, value))

            if isinstance(value, dict):
                return json.dumps(value, ensure_ascii=False)

            return str(value)

    return ""


def build_input(claim, evidence, verdict, justification):
    return (
        f"Claim: {claim}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}"
        f"Evidence: {evidence}\n"
    )


def build_input_no_evidence(claim, evidence, verdict, justification):
    return (
        f"Claim: {claim}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}"
    )


def get_parameter_counts(model):
    trainable_params = 0
    all_params = 0

    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    return all_params, trainable_params


def print_trainable_parameters(model):
    all_params, trainable_params = get_parameter_counts(model)

    print(
        f"trainable params: {trainable_params:,} || "
        f"all params: {all_params:,} || "
        f"trainable%: {100 * trainable_params / all_params:.2f}"
    )


def get_file_size_mb(path):
    if path is None or not os.path.exists(path):
        return np.nan
    return os.path.getsize(path) / (1024 ** 2)


def sync_if_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def get_hardware_name():
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    return "CPU"

def clean_reasoning_trace(trace):
    justification = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting|SUPPORTS|REFUTES|CONFLICTING|Supports|Refutes))",
        "",
        trace,
        flags=re.IGNORECASE,
    )
    justification = justification.strip().replace("\n", " ").split("Label:")[0]
    return justification

In [9]:
# =========================
# 5. Dataset
# =========================

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["model_input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item


In [8]:
# =========================
# 6. Roberta verifier model
# =========================

class CustomClassifier(torch.nn.Module):
    def __init__(
            self,
            model_name,
            num_labels=1,
            hidden_dim=None,
            dropout_value=0.1,
            freeze_base_layer=False,
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(model_name)

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = pooled_output.float()

        logits = self.classifier(pooled_output)
        return logits

In [12]:
# =========================
# 7. Trainer
# =========================

class TrainerModule:
    def __init__(
            self,
            model,
            tokenizer,
            train_loader,
            val_loader,
            epochs,
            lr,
            output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.tokenizer = tokenizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()

            total_loss = 0
            total_acc = 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask).squeeze(-1)
                logits = torch.clamp(logits, -50, 50)
                labels = labels.float()
                if torch.isnan(logits).any():
                  print("NaN detected — debugging batch")
                  print("input_ids max:", input_ids.max().item())
                  print("input_ids min:", input_ids.min().item())
                  print("attention_mask sum:", attention_mask.sum().item())
                  print("example text:", self.train_loader.dataset.texts[0][:500])
                  break
                assert not torch.isnan(logits).any(), "NaN in logits"
                loss = self.loss_fn(logits, labels)
                assert not torch.isnan(loss), "NaN loss"


                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                preds = (
                        torch.sigmoid(logits) >= 0.5
                ).detach().cpu().numpy()

                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()

        total_loss = 0
        total_acc = 0

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask).squeeze(-1)
                logits = torch.clamp(logits, -50, 50)
                labels = labels.float()
                assert not torch.isnan(logits).any(), "NaN in logits"
                loss = self.loss_fn(logits, labels)
                assert not torch.isnan(loss), "NaN loss"

                total_loss += loss.item()

                preds = (
                        torch.sigmoid(logits) >= 0.5
                ).detach().cpu().numpy()

                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")

        self.tokenizer.save_pretrained(self.output_dir)

        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"model_epoch_{epoch}.pt"),
        )


In [11]:
# =========================
# 8. Prediction evaluator
# =========================

class VerifierEvaluator:
    def __init__(
            self,
            model_path,
            tokenizer_path,
            base_model,
            device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

        self.model = CustomClassifier(
            model_name=base_model,
            freeze_base_layer=False,
        )

        self.model.load_state_dict(
            torch.load(model_path, map_location=self.device)
        )

        self.model.to(self.device)
        self.model.eval()

    def encode_input(
            self,
            claim,
            evidence,
            verdict,
            justification,
            max_length=MAX_LENGTH,
    ):
        text = build_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, evidence, verdict, justification):
        input_ids, attention_mask = self.encode_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        with torch.no_grad():
            score = self.model(input_ids, attention_mask).item()

        return float(score)

In [ ]:

# =========================
# 9. Main experiment loop
# =========================

all_results = []

hardware_name = get_hardware_name()
run_on_hardware = "GPU" if torch.cuda.is_available() else "CPU"

for lang_cfg in LANGUAGES:
    lang = lang_cfg["lang"]

    print(f"\n{'='*80}")
    print(f"  LANGUAGE: {lang.upper()}")
    print(f"{'='*80}")

    # ── preprocessing ─────────────────────────────────────────────────────
    if os.path.exists(lang_cfg["train_jsonl"]):
        print(f"[{lang}] JSONL exists, skipping preprocessing")
    else:
        print(f"[{lang}] Running preprocessing ...")
        subprocess.run(
            [
                sys.executable,
                PREPROCESSOR_PATH,
                "--input",  lang_cfg["train_path"],
                "--output", lang_cfg["train_jsonl"],
            ],
            cwd=BASE,
            check=True,
        )

    base_train_df = pd.read_json(lang_cfg["train_jsonl"], lines=True)
    print(f"[{lang}] {len(base_train_df):,} training examples | class dist: {dict(base_train_df['Class'].value_counts())}")

    with open(lang_cfg["val_path"], "r", encoding="utf-8") as f:
        val_data = json.load(f)
    print(f"[{lang}] {len(val_data):,} validation samples")

    for use_evidence in EVIDENCE_CONDITIONS:
        ev_tag   = "with_evidence" if use_evidence else "no_evidence"
        build_fn = build_input if use_evidence else build_input_no_evidence

        ckpt_id = f"{MODEL_ID}_{MODEL_TYPE}_{DISTILL_TAG}_{PREPROCESSING_FAMILY}_{ev_tag}_{lang}"
        MODEL_DIR = f"{CKPT_ROOT}/{lang}/{ckpt_id}"
        os.makedirs(MODEL_DIR, exist_ok=True)

        print(f"\n{'─'*80}")
        print(f"  [{lang}] Evidence condition: {ev_tag}")
        print(f"  Checkpoint dir: {MODEL_DIR}")
        print(f"{'─'*80}")

        # ── build model input text ─────────────────────────────────────────
        train_df = base_train_df.copy()
        train_df["model_input_text"] = [
            build_fn(
                claim=row["Claim"],
                evidence=row.get("Evidence", ""),
                verdict=row["Verdict"],
                justification=row["Justification"],
            )
            for _, row in train_df.iterrows()
        ]

        # ── train or skip if checkpoints exist ─────────────────────────────
        expected_ckpts = [
            os.path.join(MODEL_DIR, f"model_epoch_{e}.pt") for e in range(EPOCHS)
        ]

        training_time_sec = np.nan
        training_status = "skipped_existing_checkpoint"

        if all(os.path.exists(p) for p in expected_ckpts):
            print(f"[{lang}|{ev_tag}] All {EPOCHS} checkpoints found, skipping training")
        else:
            training_status = "trained"

            train_split, dev_split = train_test_split(
                train_df,
                test_size=0.2,
                stratify=train_df["Class"],
                random_state=RANDOM_STATE,
            )

            tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

            train_dataset = TextDataset(train_split, tokenizer, MAX_LENGTH)
            dev_dataset   = TextDataset(dev_split,   tokenizer, MAX_LENGTH)

            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE)

            model = CustomClassifier(model_name=BASE_MODEL, freeze_base_layer=False)
            print_trainable_parameters(model)

            trainer = TrainerModule(
                model=model,
                tokenizer=tokenizer,
                train_loader=train_loader,
                val_loader=dev_loader,
                epochs=EPOCHS,
                lr=LR,
                output_dir=MODEL_DIR,
            )

            sync_if_cuda()
            train_start = time.perf_counter()
            trainer.train()
            sync_if_cuda()
            train_end = time.perf_counter()

            training_time_sec = train_end - train_start

        # ── load final checkpoint ──────────────────────────────────────────
        MODEL_PATH = os.path.join(MODEL_DIR, f"model_epoch_{EPOCHS - 1}.pt")

        evaluator = VerifierEvaluator(
            model_path=MODEL_PATH,
            tokenizer_path=BASE_MODEL,
            base_model=BASE_MODEL,
        )

        nr_params, trainable_params = get_parameter_counts(evaluator.model)
        model_size_mb = get_file_size_mb(MODEL_PATH)

        print(f"[{lang}|{ev_tag}] Parameters: {nr_params:,}")
        print(f"[{lang}|{ev_tag}] Model checkpoint size: {model_size_mb:.2f} MB")
        print(f"[{lang}|{ev_tag}] Training time: {training_time_sec if not np.isnan(training_time_sec) else 'skipped'} sec")

        # ── score all validation traces once and measure latency ───────────
        print(f"[{lang}|{ev_tag}] Scoring validation traces ...")

        scored_samples = []
        total_trace_count = 0

        sync_if_cuda()
        inference_start = time.perf_counter()

        for idx, sample in enumerate(tqdm(val_data, desc=f"{lang}|{ev_tag}")):
            claim    = sample["claim"]
            evidence = get_evidence(sample)

            verdict_list = []
            score_list = []
            justification_list = []

            traces = sample["Reasoning_traces"]
            total_trace_count += len(traces)

            for trace_idx in range(len(traces)):
                justification = re.sub(
                    r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
                    "",
                    sample["Reasoning_traces"][trace_idx],
                    flags=re.IGNORECASE,
                ).strip().replace("\n", " ").split("Label:")[0]

                verdict = sample["Verdict_list"][trace_idx].lower()

                inp = build_fn(claim, evidence, verdict, justification)

                enc = evaluator.tokenizer(
                    inp,
                    return_tensors="pt",
                    max_length=MAX_LENGTH,
                    truncation=True,
                    padding="max_length",
                )

                with torch.no_grad():
                    logits = evaluator.model(
                        enc["input_ids"].to(device),
                        enc["attention_mask"].to(device),
                    )

                score = logits[0][0].item()

                verdict_list.append(sample["Verdict_list"][trace_idx])
                justification_list.append(justification)
                score_list.append(score)

            scored_samples.append({
                "query_id"          : sample.get("query_id", idx),
                "Claim"             : claim,
                "Evidence"          : evidence,
                "Label"             : sample["label"],
                "verdict_list"      : verdict_list,
                "score_list"        : score_list,
                "justification_list": justification_list,
            })

        sync_if_cuda()
        inference_end = time.perf_counter()

        inference_time_sec = inference_end - inference_start
        val_avg_time_claim_sec = inference_time_sec / max(len(val_data), 1)
        val_avg_time_reasoning_sec = inference_time_sec / max(total_trace_count, 1)

        val_avg_time_claim_ms = val_avg_time_claim_sec * 1000
        val_avg_time_reasoning_ms = val_avg_time_reasoning_sec * 1000

        print(f"[{lang}|{ev_tag}] Total validation inference time: {inference_time_sec:.2f} sec")
        print(f"[{lang}|{ev_tag}] Val avg time / claim: {val_avg_time_claim_ms:.2f} ms")
        print(f"[{lang}|{ev_tag}] Val avg time / reasoning trace: {val_avg_time_reasoning_ms:.2f} ms")

        # ── apply each aggregation + run scorer ────────────────────────────
        for agg_name, agg_fn in AGGREGATIONS:
            run_id = f"{MODEL_ID}_{MODEL_TYPE}_{DISTILL_TAG}_{PREPROCESSING_FAMILY}_{ev_tag}_{lang}_{agg_name}"

            RESULT_DIR = f"{RESULTS_ROOT}/{lang}/{run_id}"
            PRED_DIR   = f"{PRED_ROOT}/{lang}"
            RUNS_DIR   = f"{BASE}/runs/{lang}"

            os.makedirs(RESULT_DIR, exist_ok=True)
            os.makedirs(PRED_DIR, exist_ok=True)
            os.makedirs(RUNS_DIR, exist_ok=True)

            pred_json_name = f"val_predictions_{run_id}.json"
            result_name    = f"val_result_{run_id}.csv"
            ir_name        = f"val_per_sample_ir_{run_id}.csv"
            trec_name      = f"val_trec_{run_id}.txt"

            PRED_PATH      = f"{PRED_DIR}/{pred_json_name}"
            RESULT_PATH    = f"{RESULT_DIR}/{result_name}"
            IR_PATH        = f"{RESULT_DIR}/{ir_name}"
            TREC_PATH      = f"{RUNS_DIR}/{trec_name}"

            print(f"\n  → [{lang}|{ev_tag}] aggregation: {agg_name}")
            print(f"    Run ID: {run_id}")

            predictions = []
            for s in scored_samples:
                best_verdict = agg_fn(s["verdict_list"], s["score_list"])

                predictions.append({
                    "query_id"        : s["query_id"],
                    "Claim"           : s["Claim"],
                    "Evidence"        : s["Evidence"],
                    "Label"           : s["Label"],
                    "Verdict_BoN"     : best_verdict,
                    "BoN_Verdict_list": s["verdict_list"],
                    "Reasoning_traces": s["justification_list"],
                    "score_list"      : s["score_list"],
                })

            # Save named validation prediction JSON
            with open(PRED_PATH, "w", encoding="utf-8") as fp:
                json.dump(predictions, fp, indent=4, ensure_ascii=False)

            # Scorer script is from Git repo, but files are read/written in Drive.
            # scorer.py expects: output/RM_prediction/clef_predictions.json

            print("Scorer script:", SCORER_PATH)
            print("Scorer exists:", os.path.exists(SCORER_PATH))

            shutil.copy(PRED_PATH, SCORER_INPUT)

            result = subprocess.run(
                [sys.executable, SCORER_PATH],
                cwd=BASE,
                capture_output=True,
                text=True,
            )

            print("Scorer return code:", result.returncode)
            print("Scorer stdout:")
            print(result.stdout)
            print("Scorer stderr:")
            print(result.stderr)

            result.check_returncode()

            # Copy scorer output from Drive temp folder to organized result folder
            shutil.copy(SCORER_RESULT, RESULT_PATH)
            shutil.copy(SCORER_IR, IR_PATH)

            # Parse metrics
            with open(RESULT_PATH, "r", encoding="utf-8") as f:
                content = f.read()

            m_f1 = re.search(
                r"^macro avg,[0-9.]+,[0-9.]+,([0-9.]+)",
                content,
                re.MULTILINE,
            )
            m_r5 = re.search(
                r"^5,([0-9.]+)",
                content,
                re.MULTILINE,
            )

            macro_f1 = float(m_f1.group(1)) if m_f1 else float("nan")
            recall_at5 = float(m_r5.group(1)) if m_r5 else float("nan")

            # Create TREC-style ranking file
            with open(TREC_PATH, "w", encoding="utf-8") as out:
                for sample in predictions:
                    query_id = sample["query_id"]
                    ranked = sorted(
                        enumerate(sample["score_list"]),
                        key=lambda x: x[1],
                        reverse=True,
                    )

                    for rank, (trace_idx, score) in enumerate(ranked, start=1):
                        out.write(
                            f"{query_id}\tQ0\t{query_id}_{trace_idx}\t{rank}\t{score:.6f}\t{run_id}\n"
                        )

            trec_created = os.path.exists(TREC_PATH)

            all_results.append({
                "Model": MODEL_NAME,
                "Type (baseline/experiment)": MODEL_TYPE,

                "English": "yes" if lang == "english" else "",
                "Spanish": "yes" if lang == "spanish" else "",
                "Arabic": "yes" if lang == "arabic" else "",

                "Language": lang,
                "Base model": BASE_MODEL,
                "Student model": BASE_MODEL,
                "Teacher model": "none",
                "Distillation strategy": DISTILLATION_STRATEGY,
                "Preprocessing": ev_tag,
                "Aggregation": agg_name,
                "Run ID": run_id,

                "Val results.csv": RESULT_PATH,
                "Val per_sample_ir.csv": IR_PATH,
                "Val prediction JSON": PRED_PATH,

                "Nr. Of parameters": nr_params,
                "Trainable parameters": trainable_params,
                "Model size (MB)": round(model_size_mb, 2),

                "Training status": training_status,
                "Training time": round(training_time_sec, 2) if not np.isnan(training_time_sec) else "",

                "Val Avg time/claim": round(val_avg_time_claim_ms, 2),
                "Val Avg. time/reasoning": round(val_avg_time_reasoning_ms, 2),

                "Val Macro F1": macro_f1,
                "Val Recall@5": recall_at5,

                "TREC file created": "yes" if trec_created else "no",
                "TREC file path": TREC_PATH if trec_created else "",

                "Run on hardware (CPU/GPU)": run_on_hardware,
                "Hardware name": hardware_name,
                "Batch Size": BATCH_SIZE,
                "Inference batch size": 1,

                "Checkpoint path": MODEL_PATH,
                "n_val_claims": len(predictions),
                "n_val_reasoning_traces": total_trace_count,
                "Val Avg traces/claim": round(total_trace_count / max(len(predictions), 1), 2),
            })

            print(f"    Val Macro F1={macro_f1:.4f} | Val Recall@5={recall_at5:.4f}")
            print(f"    result.csv: {RESULT_PATH}")
            print(f"    per_sample_ir.csv: {IR_PATH}")
            print(f"    prediction JSON: {PRED_PATH}")
            print(f"    TREC: {TREC_PATH}")


  LANGUAGE: ENGLISH
[english] JSONL exists, skipping preprocessing
[english] 31,433 training examples | class dist: {0: np.int64(22775), 1: np.int64(8658)}
[english] 1,600 validation samples

────────────────────────────────────────────────────────────────────────────────
  [english] Evidence condition: with_evidence
  Checkpoint dir: /content/drive/MyDrive/AIR_CheckThat/output/checkpoints/english/roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_english
────────────────────────────────────────────────────────────────────────────────
[english|with_evidence] All 3 checkpoints found, skipping training


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[english|with_evidence] Parameters: 124,646,401
[english|with_evidence] Model checkpoint size: 475.57 MB
[english|with_evidence] Training time: skipped sec
[english|with_evidence] Scoring validation traces ...


english|with_evidence: 100%|██████████| 1600/1600 [04:54<00:00,  5.43it/s]


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[arabic|with_evidence] Parameters: 124,646,401
[arabic|with_evidence] Model checkpoint size: 475.57 MB
[arabic|with_evidence] Training time: 937.721438732 sec
[arabic|with_evidence] Scoring validation traces ...


arabic|with_evidence: 100%|██████████| 652/652 [02:50<00:00,  3.83it/s]


[arabic|with_evidence] Total validation inference time: 170.35 sec
[arabic|with_evidence] Val avg time / claim: 261.28 ms
[arabic|with_evidence] Val avg time / reasoning trace: 13.06 ms

  → [arabic|with_evidence] aggregation: top1
    Run ID: roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_arabic_top1
Scorer script: /content/AIR_Group_Task/task2/scorer.py
Scorer exists: True
Scorer return code: 0
Scorer stdout:
Total unique claims: 644

--- IR Ranking Metrics ---
    k   Mean Recall@k  Mean Precision@k
    1          0.0424            0.6925
    2          0.0812            0.6910
    3          0.1219            0.6946
    4          0.1701            0.6988
    5          0.2102            0.6997

--- Per-Class Metrics (Verdict_BoN) ---
         false:  P=0.8142  R=0.6808  F1=0.7415  n=354
          true:  P=0.7621  R=0.7069  F1=0.7335  n=290
   conflicting:  P=0.0000  R=0.0000  F1=0.0000  n=0

--- Aggregate Metrics ---
  macro avg:  P=0.5254  R=0.4626  F1=0.4

In [ ]:
# =========================
# 10. Results summary
# =========================

results_df = pd.DataFrame(all_results)

summary_path = f"{BASE}/output/roberta_experiment_summary.csv"
results_df.to_csv(summary_path, index=False)

print("\n" + "=" * 120)
print("FULL CHECKLIST SUMMARY")
print("=" * 120)

display_cols = [
    "Model",
    "Type (baseline/experiment)",
    "Language",
    "Preprocessing",
    "Aggregation",
    "Val Macro F1",
    "Val Recall@5",
    "Nr. Of parameters",
    "Model size (MB)",
    "Training time",
    "Val Avg time/claim",
    "Val Avg. time/reasoning",
    "Val Avg traces/claim",
    "TREC file created",
    "Run on hardware (CPU/GPU)",
    "Batch Size",
]

print(results_df[display_cols].to_string(index=False))

print("\n" + "=" * 120)
print("BEST CONFIG PER LANGUAGE BY VAL MACRO F1")
print("=" * 120)

for lang, grp in results_df.groupby("Language"):
    best = grp.loc[grp["Val Macro F1"].idxmax()]
    print(
        f"{lang:<10} | "
        f"preprocessing={best['Preprocessing']:<15} | "
        f"aggregation={best['Aggregation']:<16} | "
        f"F1={best['Val Macro F1']:.4f} | "
        f"R@5={best['Val Recall@5']:.4f} | "
        f"TREC={best['TREC file path']}"
    )

print("\n" + "=" * 120)
print("RESULT FILE LOCATIONS")
print("=" * 120)

for _, row in results_df.iterrows():
    print(f"\nRun ID: {row['Run ID']}")
    print(f"  Language: {row['Language']}")
    print(f"  Val results.csv: {row['Val results.csv']}")
    print(f"  Val per_sample_ir.csv: {row['Val per_sample_ir.csv']}")
    print(f"  Val prediction JSON: {row['Val prediction JSON']}")
    print(f"  TREC file: {row['TREC file path']}")
    print(f"  Checkpoint: {row['Checkpoint path']}")

print("\n" + "=" * 120)
print(f"Saved full checklist summary to: {summary_path}")
print("=" * 120)


FULL CHECKLIST SUMMARY
  Model Type (baseline/experiment) Language Preprocessing    Aggregation  Val Macro F1  Val Recall@5  Nr. Of parameters  Model size (MB) Training time  Val Avg time/claim  Val Avg. time/reasoning  Val Avg traces/claim TREC file created Run on hardware (CPU/GPU)  Batch Size
roberta                   baseline  english with_evidence           top1        0.4784        0.2534          124646401           475.57                            184.20                    12.28                  15.0               yes                       GPU           4
roberta                   baseline  english with_evidence  majority_top3        0.4785        0.2534          124646401           475.57                            184.20                    12.28                  15.0               yes                       GPU           4
roberta                   baseline  english with_evidence  majority_top5        0.4731        0.2534          124646401           475.57                  

In [ ]:
# =========================
# 11. Copy best runs to a separate folder
# =========================

BEST_RUNS_DIR = f"{BASE}/runs/best_by_language"
os.makedirs(BEST_RUNS_DIR, exist_ok=True)

print("\n" + "=" * 80)
print("COPYING BEST RUNS PER LANGUAGE")
print("=" * 80)

for lang, grp in results_df.groupby("Language"):
    best = grp.loc[grp["Val Macro F1"].idxmax()]

    src_trec = best["TREC file path"]
    src_json = best["Val prediction JSON"]

    best_trec_name = f"BEST_{lang}_{best['Run ID']}.txt"
    best_json_name = f"BEST_{lang}_{best['Run ID']}.json"

    dst_trec = os.path.join(BEST_RUNS_DIR, best_trec_name)
    dst_json = os.path.join(BEST_RUNS_DIR, best_json_name)

    if os.path.exists(src_trec):
        shutil.copy(src_trec, dst_trec)

    if os.path.exists(src_json):
        shutil.copy(src_json, dst_json)

    print(f"\n[{lang.upper()}]")
    print(f"  Best run ID: {best['Run ID']}")
    print(f"  F1={best['Val Macro F1']:.4f} | R@5={best['Val Recall@5']:.4f}")
    print(f"  Best TREC copied to: {dst_trec}")
    print(f"  Best JSON copied to: {dst_json}")


COPYING BEST RUNS PER LANGUAGE

[ARABIC]
  Best run ID: roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_arabic_score_weighted
  F1=0.4976 | R@5=0.2102
  Best TREC copied to: /content/drive/MyDrive/AIR_CheckThat/runs/best_by_language/BEST_arabic_roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_arabic_score_weighted.txt
  Best JSON copied to: /content/drive/MyDrive/AIR_CheckThat/runs/best_by_language/BEST_arabic_roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_arabic_score_weighted.json

[ENGLISH]
  Best run ID: roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_english_score_weighted
  F1=0.4833 | R@5=0.2534
  Best TREC copied to: /content/drive/MyDrive/AIR_CheckThat/runs/best_by_language/BEST_english_roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_english_score_weighted.txt
  Best JSON copied to: /content/drive/MyDrive/AIR_CheckThat/runs/best_by_language/BEST_english_roberta_b

In [ ]:
# =========================
# 12. Prepare for run on test data
# =========================
def run_test_prediction_for_language(
    lang,
    test_path,
    checkpoint_path,
    aggregation_name,
    aggregation_fn,
):
    print("\n" + "=" * 100)
    print(f"RUNNING TEST PREDICTION: {lang.upper()} | aggregation={aggregation_name}")
    print("=" * 100)

    if not os.path.exists(test_path):
        raise FileNotFoundError(f"Test file not found: {test_path}")

    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    with open(test_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)

    print(f"[{lang}] Test samples: {len(test_data)}")
    print(f"[{lang}] Checkpoint: {checkpoint_path}")

    evaluator = VerifierEvaluator(
        model_path=checkpoint_path,
        tokenizer_path=BASE_MODEL,
        base_model=BASE_MODEL,
    )

    predictions = []
    total_trace_count = 0

    sync_if_cuda()
    start_time = time.perf_counter()

    for idx, sample in enumerate(tqdm(test_data, desc=f"test|{lang}")):
        query_id = sample.get("query_id", idx)
        claim = sample["claim"]
        evidence = get_evidence(sample)

        verdict_list = []
        score_list = []
        justification_list = []

        traces = sample["Reasoning_traces"]
        total_trace_count += len(traces)

        for trace_idx, trace in enumerate(traces):
            justification = clean_reasoning_trace(trace)
            verdict = sample["Verdict_list"][trace_idx].lower()

            inp = build_input(
                claim=claim,
                evidence=evidence,
                verdict=verdict,
                justification=justification,
            )

            enc = evaluator.tokenizer(
                inp,
                return_tensors="pt",
                max_length=MAX_LENGTH,
                truncation=True,
                padding="max_length",
            )

            with torch.no_grad():
                logits = evaluator.model(
                    enc["input_ids"].to(device),
                    enc["attention_mask"].to(device),
                )

            score = logits[0][0].item()

            verdict_list.append(sample["Verdict_list"][trace_idx])
            justification_list.append(justification)
            score_list.append(float(score))

        final_verdict = aggregation_fn(verdict_list, score_list)

        predictions.append({
            "query_id"        : query_id,
            "Claim"           : claim,
            "Label"           : sample.get("label", ""),  # empty if test has no gold label
            "Verdict_BoN"     : final_verdict,
            "BoN_Verdict_list": verdict_list,
            "Reasoning_traces": justification_list,
            "score_list"      : score_list,
        })

    sync_if_cuda()
    end_time = time.perf_counter()

    total_time = end_time - start_time
    avg_time_claim_ms = (total_time / max(len(test_data), 1)) * 1000
    avg_time_reasoning_ms = (total_time / max(total_trace_count, 1)) * 1000

    print(f"[{lang}] Test inference time: {total_time:.2f} sec")
    print(f"[{lang}] Avg time / claim: {avg_time_claim_ms:.2f} ms")
    print(f"[{lang}] Avg time / reasoning trace: {avg_time_reasoning_ms:.2f} ms")

    return predictions, {
        "test_samples": len(test_data),
        "test_reasoning_traces": total_trace_count,
        "test_total_time_sec": total_time,
        "test_avg_time_claim_ms": avg_time_claim_ms,
        "test_avg_time_reasoning_ms": avg_time_reasoning_ms,
    }

import zipfile

def save_codabench_file(predictions, lang, output_dir, task_prefix="Task2"):
    os.makedirs(output_dir, exist_ok=True)

    lang_name_map = {
        "english": "English",
        "spanish": "Spanish",
        "arabic": "Arabic",
    }

    lang_name = lang_name_map[lang]

    json_name = f"{task_prefix}_Numerical_claims_{lang_name}.json"
    zip_name  = f"{task_prefix}_Numerical_claims_{lang_name}.zip"

    json_path = os.path.join(output_dir, json_name)
    zip_path  = os.path.join(output_dir, zip_name)

    clean_predictions = []

    for item in predictions:
        clean_predictions.append({
            "query_id"        : item["query_id"],
            "Claim"           : item["Claim"],
            "Label"           : item.get("Label", ""),
            "Verdict_BoN"     : item["Verdict_BoN"],
            "BoN_Verdict_list": item["BoN_Verdict_list"],
            "Reasoning_traces": item["Reasoning_traces"],
            "score_list"      : item["score_list"],
        })

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(clean_predictions, f, indent=4, ensure_ascii=False)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(json_path, arcname=json_name)

    print(f"Saved JSON: {json_path}")
    print(f"Saved ZIP : {zip_path}")

    return json_path, zip_path


In [ ]:
# =========================
# 13. Run best validation configuration on test
# =========================

summary_path = f"{BASE}/output/roberta_experiment_summary.csv"
results_df = pd.read_csv(summary_path)

SUBMISSION_DIR = (
    f"{BASE}/codabench_submissions/"
    f"{MODEL_NAME}_{MODEL_TYPE}_{DISTILL_TAG}_{PREPROCESSING_FAMILY}_best_val_config"
)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

AGG_FN_MAP = {
    "top1": agg_top1,
    "majority_top3": agg_majority_top3,
    "majority_top5": agg_majority_top5,
    "score_weighted": agg_score_weighted,
}

test_summary_rows = []

for lang_cfg in LANGUAGES:
    lang = lang_cfg["lang"]

    if "test_path" not in lang_cfg:
        raise KeyError(f"Missing test_path for {lang}. Add test_path to LANGUAGES.")

    test_path = lang_cfg["test_path"]

    lang_results = results_df[results_df["Language"] == lang].copy()

    if len(lang_results) == 0:
        raise ValueError(f"No validation results found for language: {lang}")

    best = lang_results.loc[lang_results["Val Macro F1"].idxmax()]

    best_agg_name = best["Aggregation"]
    best_agg_fn = AGG_FN_MAP[best_agg_name]
    checkpoint_path = best["Checkpoint path"]

    print("\n" + "=" * 100)
    print(f"BEST VAL CONFIG FOR {lang.upper()}")
    print("=" * 100)
    print(f"Aggregation: {best_agg_name}")
    print(f"Val Macro F1: {best['Val Macro F1']}")
    print(f"Val Recall@5: {best['Val Recall@5']}")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Test path: {test_path}")

    test_predictions, test_metrics = run_test_prediction_for_language(
        lang=lang,
        test_path=test_path,
        checkpoint_path=checkpoint_path,
        aggregation_name=best_agg_name,
        aggregation_fn=best_agg_fn,
    )

    json_path, zip_path = save_codabench_file(
        predictions=test_predictions,
        lang=lang,
        output_dir=SUBMISSION_DIR,
        task_prefix="Task2",
    )

    test_summary_rows.append({
        "Language": lang,
        "Model": MODEL_NAME,
        "Model type": MODEL_TYPE,
        "Base model": BASE_MODEL,
        "Distillation strategy": DISTILLATION_STRATEGY,
        "Preprocessing family": PREPROCESSING_FAMILY,
        "Evidence input": "with_evidence",
        "Run ID": best["Run ID"],
        "Selected aggregation": best_agg_name,
        "Selected by Val Macro F1": best["Val Macro F1"],
        "Selected Val Recall@5": best["Val Recall@5"],
        "Checkpoint path": checkpoint_path,
        "Test path": test_path,
        "Test prediction JSON": json_path,
        "Test submission ZIP": zip_path,
        "Test samples": test_metrics["test_samples"],
        "Test reasoning traces": test_metrics["test_reasoning_traces"],
        "Test total time sec": round(test_metrics["test_total_time_sec"], 2),
        "Test avg time/claim ms": round(test_metrics["test_avg_time_claim_ms"], 2),
        "Test avg time/reasoning ms": round(test_metrics["test_avg_time_reasoning_ms"], 2),
    })

test_summary_df = pd.DataFrame(test_summary_rows)
test_summary_path = f"{SUBMISSION_DIR}/roberta_test_submission_summary.csv"
test_summary_df.to_csv(test_summary_path, index=False)

print("\n" + "=" * 120)
print("TEST SUBMISSION SUMMARY")
print("=" * 120)
print(test_summary_df.to_string(index=False))
print("=" * 120)
print(f"Saved test summary to: {test_summary_path}")


BEST VAL CONFIG FOR ENGLISH
Aggregation: score_weighted
Val Macro F1: 0.4833
Val Recall@5: 0.2534
Checkpoint: /content/drive/MyDrive/AIR_CheckThat/output/checkpoints/english/roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_english/model_epoch_2.pt
Test path: /content/drive/MyDrive/AIR_CheckThat/data/english/clef_2026_final_english_test.json

RUNNING TEST PREDICTION: ENGLISH | aggregation=score_weighted
[english] Test samples: 2558
[english] Checkpoint: /content/drive/MyDrive/AIR_CheckThat/output/checkpoints/english/roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_english/model_epoch_2.pt


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
test|english: 100%|██████████| 2558/2558 [15:53<00:00,  2.68it/s]


[english] Test inference time: 953.11 sec
[english] Avg time / claim: 372.60 ms
[english] Avg time / reasoning trace: 18.63 ms
Saved JSON: /content/drive/MyDrive/AIR_CheckThat/codabench_submissions/roberta_baseline_none_custom_evidence_preprocessing_best_val_config/Task2_Numerical_claims_English.json
Saved ZIP : /content/drive/MyDrive/AIR_CheckThat/codabench_submissions/roberta_baseline_none_custom_evidence_preprocessing_best_val_config/Task2_Numerical_claims_English.zip

BEST VAL CONFIG FOR SPANISH
Aggregation: top1
Val Macro F1: 0.3663
Val Recall@5: 0.2019
Checkpoint: /content/drive/MyDrive/AIR_CheckThat/output/checkpoints/spanish/roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_spanish/model_epoch_2.pt
Test path: /content/drive/MyDrive/AIR_CheckThat/data/spanish/clef_spanish_test_final.json

RUNNING TEST PREDICTION: SPANISH | aggregation=top1
[spanish] Test samples: 1164
[spanish] Checkpoint: /content/drive/MyDrive/AIR_CheckThat/output/checkpoints/spanish/rober

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
test|spanish: 100%|██████████| 1164/1164 [07:51<00:00,  2.47it/s]


[spanish] Test inference time: 471.79 sec
[spanish] Avg time / claim: 405.32 ms
[spanish] Avg time / reasoning trace: 20.27 ms
Saved JSON: /content/drive/MyDrive/AIR_CheckThat/codabench_submissions/roberta_baseline_none_custom_evidence_preprocessing_best_val_config/Task2_Numerical_claims_Spanish.json
Saved ZIP : /content/drive/MyDrive/AIR_CheckThat/codabench_submissions/roberta_baseline_none_custom_evidence_preprocessing_best_val_config/Task2_Numerical_claims_Spanish.zip

BEST VAL CONFIG FOR ARABIC
Aggregation: score_weighted
Val Macro F1: 0.4976
Val Recall@5: 0.2102
Checkpoint: /content/drive/MyDrive/AIR_CheckThat/output/checkpoints/arabic/roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_arabic/model_epoch_2.pt
Test path: /content/drive/MyDrive/AIR_CheckThat/data/arabic/clef_2026_final_arabic_test.json

RUNNING TEST PREDICTION: ARABIC | aggregation=score_weighted
[arabic] Test samples: 511
[arabic] Checkpoint: /content/drive/MyDrive/AIR_CheckThat/output/checkpoin

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
test|arabic: 100%|██████████| 511/511 [02:03<00:00,  4.14it/s]


[arabic] Test inference time: 123.38 sec
[arabic] Avg time / claim: 241.45 ms
[arabic] Avg time / reasoning trace: 12.07 ms
Saved JSON: /content/drive/MyDrive/AIR_CheckThat/codabench_submissions/roberta_baseline_none_custom_evidence_preprocessing_best_val_config/Task2_Numerical_claims_Arabic.json
Saved ZIP : /content/drive/MyDrive/AIR_CheckThat/codabench_submissions/roberta_baseline_none_custom_evidence_preprocessing_best_val_config/Task2_Numerical_claims_Arabic.zip

TEST SUBMISSION SUMMARY
Language   Model Model type   Base model Distillation strategy          Preprocessing family Evidence input                                                                                        Run ID Selected aggregation  Selected by Val Macro F1  Selected Val Recall@5                                                                                                                                                 Checkpoint path                                                                         

DEBUGGING

In [13]:
# =========================
# DEBUG: load existing RoBERTa-with-evidence model
# =========================

lang = "english"   # change to "spanish" or "arabic" later

summary_path = f"{BASE}/output/roberta_experiment_summary.csv"
results_df = pd.read_csv(summary_path)

lang_results = results_df[results_df["Language"] == lang].copy()
print(lang_results[["Aggregation", "Val Macro F1", "Val Recall@5", "Checkpoint path"]])

best = lang_results.loc[lang_results["Val Macro F1"].idxmax()]

checkpoint_path = best["Checkpoint path"]
best_agg = best["Aggregation"]

print("\nUsing:")
print("Language:", lang)
print("Best aggregation:", best_agg)
print("Checkpoint:", checkpoint_path)
print("Checkpoint exists:", os.path.exists(checkpoint_path))

evaluator = VerifierEvaluator(
    model_path=checkpoint_path,
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
)

      Aggregation  Val Macro F1  Val Recall@5  \
0            top1        0.4784        0.2534   
1   majority_top3        0.4785        0.2534   
2   majority_top5        0.4731        0.2534   
3  score_weighted        0.4833        0.2534   

                                     Checkpoint path  
0  /content/drive/MyDrive/AIR_CheckThat/output/ch...  
1  /content/drive/MyDrive/AIR_CheckThat/output/ch...  
2  /content/drive/MyDrive/AIR_CheckThat/output/ch...  
3  /content/drive/MyDrive/AIR_CheckThat/output/ch...  

Using:
Language: english
Best aggregation: score_weighted
Checkpoint: /content/drive/MyDrive/AIR_CheckThat/output/checkpoints/english/roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_english/model_epoch_2.pt
Checkpoint exists: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
test_path = [x for x in LANGUAGES if x["lang"] == lang][0]["test_path"]

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Test path:", test_path)
print("Test samples:", len(test_data))

Test path: /content/drive/MyDrive/AIR_CheckThat/data/english/clef_2026_final_english_test.json
Test samples: 2558


In [15]:
# =========================
# DEBUG: check first 5 queries
# =========================

for qid in [0, 1, 2, 3, 4]:
    sample = test_data[qid]
    claim = sample["claim"]
    evidence = get_evidence(sample)

    scores = []
    decoded_set = set()

    print("\n" + "=" * 100)
    print("QUERY:", qid)
    print("CLAIM:", claim)
    print("N traces:", len(sample["Reasoning_traces"]))

    for i in range(min(5, len(sample["Reasoning_traces"]))):
        justification = clean_reasoning_trace(sample["Reasoning_traces"][i])
        verdict = sample["Verdict_list"][i].lower()

        input_text = build_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        enc = evaluator.tokenizer(
            input_text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        decoded = evaluator.tokenizer.decode(
            enc["input_ids"][0],
            skip_special_tokens=True,
        )

        score = evaluator.score(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        decoded_set.add(decoded)
        scores.append(score)

        print("-" * 80)
        print("trace:", i)
        print("verdict:", verdict)
        print("score:", score)
        print("decoded input:")
        print(decoded[:1000])

    print("\nUnique decoded inputs:", len(decoded_set))
    print("Scores:", scores)


QUERY: 0
CLAIM: Indonesia did not unveil new 300,000 rupiah banknote 
N traces: 20
--------------------------------------------------------------------------------
trace: 0
verdict: true
score: -0.34798499941825867
decoded input:
Claim: Indonesia did not unveil new 300,000 rupiah banknote 
Evidence: First ever notes issued by Bank Indonesia (1953) · 1 January 1954: all "Nederlandsch Indie" government money withdrawn: 1/2, 1, and 2½ gulden notes, all dating ... Original image courtesy of Bank of Indonesia. Note image courtesy of Bank of Indonesia. Several months ago, when Indonesia announced a new series of four coins and seven bank notes, ranging in denomination from 1,000 to 100,000 rupiah, and that it would be dedicated to revered official national heroes, who
--------------------------------------------------------------------------------
trace: 1
verdict: true
score: -0.34798499941825867
decoded input:
Claim: Indonesia did not unveil new 300,000 rupiah banknote 
Evidence: First ev

In [16]:
# =========================
# DEBUG: compare old vs reordered input format on small test set
# =========================

DEBUG_LANG = "english"
DEBUG_N_QUERIES = 5
DEBUG_N_TRACES = 5

# Load best checkpoint from completed RoBERTa-with-evidence run
summary_path = f"{BASE}/output/roberta_experiment_summary.csv"
results_df = pd.read_csv(summary_path)

lang_results = results_df[results_df["Language"] == DEBUG_LANG].copy()
best = lang_results.loc[lang_results["Val Macro F1"].idxmax()]

checkpoint_path = best["Checkpoint path"]

print("Language:", DEBUG_LANG)
print("Best validation aggregation:", best["Aggregation"])
print("Best validation F1:", best["Val Macro F1"])
print("Checkpoint:", checkpoint_path)
print("Checkpoint exists:", os.path.exists(checkpoint_path))

evaluator = VerifierEvaluator(
    model_path=checkpoint_path,
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
)

# Load test data
test_path = [x for x in LANGUAGES if x["lang"] == DEBUG_LANG][0]["test_path"]

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Test path:", test_path)
print("Test samples:", len(test_data))


# Original order used before:
# Claim + Evidence + Verdict + Justification
def build_input_old_order(claim, evidence, verdict, justification):
    return (
        f"Claim: {claim}\n"
        f"Evidence: {evidence}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}"
    )


# Reordered version:
# Claim + Verdict + Justification + Evidence
def build_input_reordered(claim, evidence, verdict, justification):
    return (
        f"Claim: {claim}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}\n"
        f"Evidence: {evidence}"
    )


def score_custom_text(input_text):
    enc = evaluator.tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )

    with torch.no_grad():
        logits = evaluator.model(
            enc["input_ids"].to(device),
            enc["attention_mask"].to(device),
        )

    decoded = evaluator.tokenizer.decode(
        enc["input_ids"][0],
        skip_special_tokens=True,
    )

    return float(logits[0][0].item()), decoded


for qid in range(DEBUG_N_QUERIES):
    sample = test_data[qid]

    claim = sample["claim"]
    evidence = get_evidence(sample)

    old_scores = []
    new_scores = []

    old_decoded_set = set()
    new_decoded_set = set()

    print("\n" + "=" * 120)
    print(f"QUERY {qid}")
    print("CLAIM:", claim)
    print("N traces:", len(sample["Reasoning_traces"]))

    for trace_idx in range(min(DEBUG_N_TRACES, len(sample["Reasoning_traces"]))):
        justification = clean_reasoning_trace(sample["Reasoning_traces"][trace_idx])
        verdict = sample["Verdict_list"][trace_idx].lower()

        old_text = build_input_old_order(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        new_text = build_input_reordered(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        old_score, old_decoded = score_custom_text(old_text)
        new_score, new_decoded = score_custom_text(new_text)

        old_scores.append(old_score)
        new_scores.append(new_score)

        old_decoded_set.add(old_decoded)
        new_decoded_set.add(new_decoded)

        print("-" * 100)
        print(f"trace={trace_idx} | verdict={verdict}")
        print(f"OLD score={old_score:.6f}")
        print(f"NEW score={new_score:.6f}")

        if trace_idx == 0:
            print("\nOLD decoded input preview:")
            print(old_decoded[:800])

            print("\nNEW decoded input preview:")
            print(new_decoded[:800])

    print("\nSummary for query:", qid)
    print("OLD unique decoded inputs:", len(old_decoded_set))
    print("NEW unique decoded inputs:", len(new_decoded_set))
    print("OLD scores:", old_scores)
    print("NEW scores:", new_scores)

Language: english
Best validation aggregation: score_weighted
Best validation F1: 0.4833
Checkpoint: /content/drive/MyDrive/AIR_CheckThat/output/checkpoints/english/roberta_base_baseline_none_custom_evidence_preprocessing_with_evidence_english/model_epoch_2.pt
Checkpoint exists: True


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Test path: /content/drive/MyDrive/AIR_CheckThat/data/english/clef_2026_final_english_test.json
Test samples: 2558

QUERY 0
CLAIM: Indonesia did not unveil new 300,000 rupiah banknote 
N traces: 20
----------------------------------------------------------------------------------------------------
trace=0 | verdict=true
OLD score=-0.347985
NEW score=-5.252645

OLD decoded input preview:
Claim: Indonesia did not unveil new 300,000 rupiah banknote 
Evidence: First ever notes issued by Bank Indonesia (1953) · 1 January 1954: all "Nederlandsch Indie" government money withdrawn: 1/2, 1, and 2½ gulden notes, all dating ... Original image courtesy of Bank of Indonesia. Note image courtesy of Bank of Indonesia. Several months ago, when Indonesia announced a new series of four coins and seven bank notes, ranging in denomination from 1,000 to 100,000 rupiah, and that it would be dedicated to revered official national heroes, who

NEW decoded input preview:
Claim: Indonesia did not unveil new 300,

In [18]:
sample = test_data[0]
trace_idx = 0

claim = sample["claim"]
evidence = " ".join(map(str, sample.get("evidences", [])))
verdict = sample["Verdict_list"][trace_idx].lower()

justification = re.sub(
    r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting|SUPPORTS|REFUTES|CONFLICTING|Supports|Refutes))",
    "",
    sample["Reasoning_traces"][trace_idx],
    flags=re.IGNORECASE,
).strip().replace("\n", " ").split("Label:")[0]

inp = (
    f"Claim: {claim}\n"
    f"Verdict: {verdict}\n"
    f"Justification: {justification}\n"
    f"Evidence: {evidence}"
)

enc = evaluator.tokenizer(
    inp,
    return_tensors="pt",
    max_length=MAX_LENGTH,
    truncation=True,
    padding="max_length",
)

decoded = evaluator.tokenizer.decode(
    enc["input_ids"][0],
    skip_special_tokens=True,
)

print("RAW INPUT:")
print(inp[:2000])

print("\n" + "="*100)
print("MODEL ACTUALLY SEES AFTER TOKENIZATION/TRUNCATION:")
print(decoded)

print("\nTOKENS USED:", int(enc["attention_mask"][0].sum()), "/", MAX_LENGTH)

RAW INPUT:
Claim: Indonesia did not unveil new 300,000 rupiah banknote 
Verdict: true
Justification: The claim states that Indonesia did not unveil a new 300,000 rupiah banknote. However, the evidence provided discusses various denominations of Indonesian currency and includes specific references to the issuance of new banknotes, including a mention of limited edition notes. Importantly, it mentions that Bank Indonesia has not issued certain rumored banknotes and emphasizes that no new denomination such as a 300,000 rupiah note exists or has been announced. Therefore, based on this information from the evidence, it supports the claim that Indonesia indeed did not unveil a 300,000 rupiah banknote.  
Evidence: First ever notes issued by Bank Indonesia (1953) · 1 January 1954: all "Nederlandsch Indie" government money withdrawn: 1/2, 1, and 2½ gulden notes, all dating ... Original image courtesy of Bank of Indonesia. Note image courtesy of Bank of Indonesia. Several months ago, when Indon